# Lab: Design Your Own MLP
## Purpose:
- Implement neural networks from scratch with Keras.
- Understand how to interpret loss curves for the training process.

### Topics:
- Keras
- Loss curves
- Plotting loss curves

### Steps
* Implement a function that defines an MLP in Keras.
* Train your MLP on the embeddings dataset.
* Plot the training and test loss curve.
* Visualize the decision boundary of the model.

Date: 2026-02-21

Source: https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_3/gdm_lab_3_4_design_your_own_mlp.ipynb

References: https://github.com/google-deepmind/ai-foundations
- GDM GH repo used in AI training courses at the university & college level.

In [ ]:
%%capture
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os       # For adjusting Keras settings.
os.environ['KERAS_BACKEND'] = 'jax' # Set a parameter for Keras.

# Packages used.
import jax.numpy as jnp         # For defining matrices.
import keras                    # Defining the MLP.
import pandas as pd             # For loading the dataset.
# For splitting data into train and test splits.
from sklearn import model_selection

from ai_foundations import machine_learning # For training your MLP.
from ai_foundations import visualizations # For visualizing data and boundaries.
from ai_foundations import training # For logging the loss during training.
from ai_foundations.feedback import course_3 # For providing feedback.

# Generate the data

Load the embeddings and labels.

This cell contains a line that automatically splits the data into training and testing sets:
- Remember that the original dataset was split into [:, -1] (the prompt) and [1, :] (the target).

```python
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42
)
```

1. Takes the original embeddings (the matrix `X`) and the labels (the vector `y`)
2. Randomly splits the dataset such that 80% of the examples are in the training set and 20% are in the test set.
3. Use the test set to ensure model properly generalizes and doesn't overfit.

In [ ]:
# Load data using pandas.
df = pd.read_csv("https://storage.googleapis.com/dm-educational/assets/ai_foundations/mat-apple-bank-dataset.csv")

# Extract embeddings (Embedding_dim_1, Embedding_dim_2) and labels.
X = jnp.array(df[["Embedding_dim_1", "Embedding_dim_2"]].values)
# Labels: 0 ("mat"), 1 ("apple"), or 2 ("bank").
y = jnp.array(df["Label"].values)

# Human-readable labels.
labels = ["mat", "apple", "bank"]

# Randomly split dataset into training and test sets.
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert data into JAX arrays so that they can be used to train your MLP model.
X_train, y_train = jnp.array(X_train), jnp.array(y_train)
X_test, y_test = jnp.array(X_test), jnp.array(y_test)

## Implement the MLP

Use [Keras](https://keras.io) to implement a neural network by specifying its structure.

Definie a list of operations in the cell below. An operation may involve computing the weighted sum or applying a non-linear activation function.

Ex. to build a simple single-layer neural network, you could define a weighted sum operation and a 5-class SoftMax classification:

```python
operations = []
# keras.layers.Dense adds a weighted sum with output dimension 5 to the network.
operations.append(keras.layers.Dense(5))
# This adds the SoftMax function to be applied to the output of the previous operation.
operations.append(keras.layers.Softmax())
```

------

>`construct_operations()` takes a list of integers called `hidden_dims`.
> Each element $d$ in the list indicates that there should be $d$ neurons that compute $d$ weighted sums, followed by a ReLU activation function.
> There should be `n_classes` neurons in the output layer followed by a SoftMax operation to output a probability distribution.
> The output dimension of the network (the number of values it will output), is `n_classes`.
>
>The function returns a list of operations to pass to `build_mlp()`.
> `build_mlp()` combines the list of operations to form one model.
>
> For example, given the argument `hidden_dims=[5,10]` and `n_classes=3` it should generate a list of six operations:
>
>1. Weighted sums of 5 neurons
>2. A ReLU function
>3. Weighted sums of 10 neurons
>4. A ReLU function
>5. Weighted sums of 3 neurons
>6. A SoftMax function
>
>`keras` functions to use
> * `keras.layers.Dense(dim)`: ([docs](https://keras.io/api/layers/core_layers/dense/)) to create a layer with `dim` neurons that computes the weighted sums
>* `keras.layers.ReLU` ([docs](https://keras.io/api/layers/activation_layers/relu/)) to apply the ReLU activation function to the previous operation
>* `keras.layers.Softmax` ([docs](https://keras.io/api/layers/activation_layers/softmax)) to apply the SoftMax activation function the previous operation
>
------

In [ ]:
def construct_operations(
    hidden_dims: list[int], n_classes: int
) -> list[keras.Layer]:
    """
    A function that builds all operations, i.e., weighted sum computations
    and activation functions for an MLP with a SoftMax layer as the output
    layer.

    Args:
      hidden_dims: A list of dimensions for all hidden layers.
      n_classses: Number of classes for the output layer.

    Returns:
      A list of operations in the form of Keras Layer instances.
    """
    operations = []
    # defines each layer with i dimensions,
    # then applies the ReLU function to that layer.
    for i in hidden_dims:
        # keras.layers.Dense creates a layer with i neurons
        operations.append(keras.layers.Dense(i))
        # applies the ReLU activation function
        # NOTE: ReLU does not take an argument.
        operations.append(keras.layers.ReLU())
    # Once the layers are in place,
    # Add the weighted sum with n dimensions to the network.
    operations.append(keras.layers.Dense(n_classes))
    # Apply the SoftMax function to the output.
    operations.append(keras.layers.Softmax(n_classes))

    return operations


# Constructs the MLP from a list of operations.
# You do not need to edit this function.
def build_mlp(hidden_dims: list[int], n_classes: int) -> keras.Model:
    """
    Builds an MLP using the construct_operations function above.

    Args:
      hidden_dims: A list of dimensions for all hidden layers.
      n_classses: Number of classes for the output layer.

    Returns:
      A keras model that sequentially passes the data through all operations
        defined in opertations.
    """
    operations = construct_operations(hidden_dims, n_classes)
    return keras.Sequential(operations)

## Train your MLP
Instantiate a model.
Modify the number of hidden layers and the number of neurons in each layer.
Since the dataset defines a three-way classification task, keep the n_classes parameter set to 3.

In [ ]:
# Define a model with one hidden layer that consist of 10 neurons.
model = build_mlp([10], 3)

traning_logger = training.CustomAccuracyPrinter(print_every=20)
training_history = machine_learning.train_mlp(
    model,
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=200,
    callbacks=[traning_logger],
)

## Plot loss curves

In [ ]:
visualizations.plot_loss_curve(training_history.history)

### Visualize decision boundaries

In [ ]:
visualizations.plot_data_and_mlp(
    X_train,
    y_train,
    labels,
    features_test=X_test,
    label_ids_test=y_test,
    mlp_model=model,
    title="Decision Boundaries - Your MLP",
)